In [1]:
#Montar mi google drive para poder usar los archivos
from google.colab import drive
drive.mount('/content/gdrive')

MessageError: Error: credential propagation was unsuccessful

In [2]:




def antiSpace(strParam):
  result = ''
  for ch in strParam:
    if ch != ' ' and ch != ',' and ch != '/' and ch != '1':
      result = result + ch
    else:
      result = result + '_'
  return result

In [3]:
%cd /content/gdrive/My Drive/Ciencias/Data/WoS/InCites/ESI/ManuscriptExperiments2026/OpenAlex/GlobalContext
!ls

[Errno 2] No such file or directory: '/content/gdrive/My Drive/Ciencias/Data/WoS/InCites/ESI/ManuscriptExperiments2026/OpenAlex/GlobalContext'
/content
sample_data


In [4]:

import pandas as pd
import numpy as np
import os
#
INDICADOR = 'Web of Science Documents'
YEAR = 'Publication Year'


In [5]:
import pandas as pd
#Producción mundial: Año x Areas
df_areas = pd.read_csv('../world_production.csv', index_col=0)
df_areas

FileNotFoundError: [Errno 2] No such file or directory: '../world_production.csv'

In [ ]:
# Reading a list from a file
units = []
with open("../units_production/units.txt", "r") as f:
    for line in f:
        units.append(line.strip()) # .strip() removes leading/trailing whitespace, including newline

print(units)

In [ ]:
len(units)

In [ ]:
#Diccionario con una dataframe de produccion Años x Areas por cada unidad
df_units = {}
#Diccionario con la producción anual de los paises en un dataframe (nombre del índice 'Publication Year', columna 'Documents')
df_units_production = {}
for unit in units:
  df_units[unit] = pd.read_csv('../units_production/'+unit+'.csv', index_col=0)
  df_units_production[unit] = pd.read_csv('../units_production/'+unit+'_totalproduction.csv', index_col=0)
df_units[unit]

In [ ]:
df_units_production['Baseline for All Items'] = pd.read_csv('../units_production/Baseline for All Items_totalproduction.csv', index_col=0)

In [ ]:
years = df_areas.index
años = df_areas.index
areas = df_areas.columns
import math

def sqrtF(number):
  return math.sqrt(number)

# Indicators Calculation

In [ ]:
import numpy as np

#Usamos Baselines for all items como la PRODUCCIÓN DEL MUNDO
#Las siguientes líneas calculan la producción del mundo sumando
df_produc = pd.DataFrame(index=años, columns=['Documents'])
df_produc['Documents'] = df_units_production['Baseline for All Items']#df_baselineForAllItems['Baseline for All Items']
#df_produc.Documents = df_areas.sum(axis=1)

# Calcular el share anual de cada area OD/OW
df_areas_share = df_areas.copy()
for area in areas:
  df_areas_share[area] = df_areas[area] / df_produc.Documents # OD / OW


#Calcular el share en cada area de cada unidad OCD/OC
df_units_share={}
df_units_wshare={}
df_geometric_aggregation={}
for unit in units:
  df_units_share[unit] = df_units[unit].copy()
  df_units_wshare[unit] = df_units[unit].copy()
  df_geometric_aggregation[unit] = df_units[unit].copy()
  for area in areas:
    if area in df_units_share[unit].columns:
      df_units_share[unit][area] = df_units[unit][area] / df_units_production[unit].Documents #OCD/OC
      df_units_production[unit]['Share'] = df_units_production[unit].Documents / df_produc.Documents # Country Share
      df_units_wshare[unit][area] = df_units[unit][area] / df_areas[area] # OCD/OWD
      # Geometric aggregation, same weight (1/2)
      df_geometric_aggregation[unit][area] = np.sqrt(df_units_share[unit][area].astype(float)) + np.sqrt(df_units_wshare[unit][area].astype(float))

#Calcular el indice de actividad (AI)
df_units_AI={}
for unit in units:
  df_units_AI[unit] = df_units_share[unit].copy()
  for area in areas:
    if area in df_units_AI[unit].columns:
      df_units_AI[unit][area] = df_units_share[unit][area] / df_areas_share[area]


#Calcular F-Measure
df_units_FMeasure={}
df_units_AMeasure={}
df_units_GMeasure={}
for unit in units:
  df_units_FMeasure[unit] = df_units_share[unit].copy()
  df_units_AMeasure[unit] = df_units_share[unit].copy()
  df_units_GMeasure[unit] = df_units_share[unit].copy()
  for area in areas:
    if area in df_units_FMeasure[unit].columns:
      ODmasOC = df_areas[area] + df_units_production[unit].Documents # OD + OC
      df_units_FMeasure[unit][area] = 2*df_units[unit][area] / ODmasOC # 2*Ocd/(OD+OC)
      df_units_AMeasure[unit][area] = (df_units_share[unit][area] + df_units_wshare[unit][area])/2
      df_units_GMeasure[unit][area] = list(map(sqrtF, df_units_share[unit][area] * df_units_wshare[unit][area]))# Calculate squre root for each element


#Calcular el indice de de especialización relativa (RSI)
df_units_RSI = {}
for unit in units:
  df_units_RSI[unit] = df_units_AI[unit].copy()
  for area in areas:
    if area in df_units_RSI[unit].columns:
      df_units_RSI[unit][area] = (df_units_AI[unit][area] - 1) / (df_units_AI[unit][area] + 1)


def concatena(dataframes):
  result = pd.DataFrame(columns = np.append(['Unit Name'], areas))
  for unit in units:
    df_unit_temp = pd.DataFrame(columns = np.append(['Unit Name'], areas))
    for area in areas:
      if area in dataframes[unit].columns:
        df_unit_temp[area] = dataframes[unit][area]
    df_unit_temp['Unit Name'] = [str(e) + '_' + unit for e in dataframes[unit].index]
    frames = [df_unit_temp, result]
    result = pd.concat(frames)
  result.fillna(0, inplace=True)
  return result

#Crear un dataframe con las evoluciones de todas las unidades
df_allUnits_Share = concatena(df_units_share)
df_allUnits_Share.to_csv('share.txt', index=False, sep=';')
df_allUnits_wShare = concatena(df_units_wshare)
df_allUnits_wShare.to_csv('Wshare.txt', index=False, sep=';')
df_allUnits_AI = concatena(df_units_AI)
df_allUnits_AI.to_csv('ai.txt', index=False, sep=';')
df_allUnits_RSI = concatena(df_units_RSI)
df_allUnits_RSI.to_csv('rsi.txt', index=False, sep=';')
df_allUnits_FMeasure = concatena(df_units_FMeasure)
df_allUnits_FMeasure.to_csv('FMEASURE.txt', index=False, sep=';')
df_allUnits_GMeasure = concatena(df_units_GMeasure)
df_allUnits_GMeasure.to_csv('GMEASURE.txt', index=False, sep=';')
df_allUnits_AMeasure = concatena(df_units_AMeasure)
df_allUnits_AMeasure.to_csv('AMEASURE.txt', index=False, sep=';')
df_allUnits_GAggregation = concatena(df_geometric_aggregation)
df_allUnits_GAggregation.to_csv('GAggregation.txt', index=False, sep=';')

In [ ]:
import matplotlib.pyplot as plt
from __future__ import print_function
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
selected_unit = 'USA'

def plotShare(Unit):
  selected_unit=Unit
  dft=df_units_share[Unit]
  dft.plot(figsize=(20,10), title = Unit+' - DShareC')
  return Unit

w = interactive(plotShare, Unit=units)
display(w)

In [ ]:
import plotly.graph_objects as go

Unit = w.result
dft =df_units_share[Unit]
print(Unit)

figMex = go.Figure()
figMex.update_layout(width=1400, height=800, title= Unit+' - DShareC')
for column in dft.columns:
  figMex.add_trace( go.Scatter(x=dft.index, y=dft[column], name= column))
figMex.update_layout(
yaxis = dict(
tickfont = dict(size=20)))
figMex.show()


# Correlations between indicators - Countries are the entities

In [ ]:
listIndicators = ['Output','DShareC', 'CShareD', 'AI', 'RSI', 'FMeasure', 'Geometric Aggregtion']
listDfs = [df_units_production, df_units_share, df_units_wshare, df_units_AI, df_units_RSI, df_units_FMeasure,df_units_AMeasure, df_units_GMeasure, df_geometric_aggregation]

#Create dictionaries
dictCorrAreas = {}
dictCorrAreasDF = {}
dictCorrAreasDFAGMeans = {}
dictSpearmanCorrAreas = {}
dictSpearmanCorrAreasDF = {}
dictSpearmanCorrAreasDFAGMeans = {}


dictYearlyData = {}

#Create a dictionary in the dictionaries
for year in years:
  dictCorrAreas[year] = {}
  dictSpearmanCorrAreas[year] = {}
  dictCorrAreasDF[year] = pd.DataFrame(index = areas)
  dictCorrAreasDFAGMeans[year] = pd.DataFrame(index = areas)
  dictSpearmanCorrAreasDF[year] = pd.DataFrame(index = areas)
  dictSpearmanCorrAreasDFAGMeans[year] = pd.DataFrame(index = areas)
  dictYearlyData[year] = {}


  for area in areas:
    dfTemp = pd.DataFrame(index=units, columns = listIndicators)
    for unit in units:
      if not df_units_share[unit].empty and area in df_units_share[unit].columns and year in df_units_share[unit].index:
        if df_units[unit][area][year].astype('float') != 0:
          dfTemp.at[unit, 'Output'] = df_units[unit][area][year].astype('float')
          dfTemp.at[unit, 'DShareC'] = df_units_share[unit][area][year].astype('float')
          dfTemp.at[unit, 'CShareD'] = df_units_wshare[unit][area][year].astype('float')
          dfTemp.at[unit, 'AI'] = df_units_AI[unit][area][year].astype('float')
          dfTemp.at[unit, 'RSI'] = df_units_RSI[unit][area][year].astype('float')
          dfTemp.at[unit, 'FMeasure'] = df_units_FMeasure[unit][area][year].astype('float')
          dfTemp.at[unit, 'AMean'] = df_units_AMeasure[unit][area][year].astype('float')
          dfTemp.at[unit, 'GMean'] = df_units_GMeasure[unit][area][year].astype('float')
          dfTemp.at[unit, 'Geometric Aggregtion'] = df_geometric_aggregation[unit][area][year].astype('float')
    dfTemp = dfTemp.dropna(how='all')
    dfTemp = dfTemp.astype('float')
    ## CORRELATION COEFFICIENTS
    dictCorrAreas[year][area] = dfTemp.corr(method='pearson')
    dictSpearmanCorrAreas[year][area] = dfTemp.corr(method='spearman')
    ## TESTS AND DATA



    #dictCorrAreasDF[year].at[area,'AI vs Output'] = dictCorrAreas[year][area].at['AI','Output']
    dictCorrAreasDF[year].at[area,'Output'] =  df_areas.at[year,area]
    dictCorrAreasDF[year].at[area,'AI vs DShareC'] = dictCorrAreas[year][area].at['AI', 'DShareC']
    dictCorrAreasDF[year].at[area,'AI vs CShareD'] = dictCorrAreas[year][area].at['AI','CShareD']
    #dictCorrAreasDF[year].at[area,'F-Measure vs Output'] = dictCorrAreas[year][area].at['FMeasure','Output']
    dictCorrAreasDF[year].at[area,'F-Measure vs DShareC'] = dictCorrAreas[year][area].at['FMeasure', 'DShareC']
    dictCorrAreasDF[year].at[area,'F-Measure vs CShareD'] = dictCorrAreas[year][area].at['FMeasure', 'CShareD']
    #dictCorrAreasDF[year].at[area,'Geometric Aggregtion vs Output'] = dictCorrAreas[year][area].at['Geometric Aggregtion','Output']
    dictCorrAreasDF[year].at[area,'Geometric Aggregtion vs DShareC'] = dictCorrAreas[year][area].at['Geometric Aggregtion', 'DShareC']
    dictCorrAreasDF[year].at[area,'Geometric Aggregtion vs CShareD'] = dictCorrAreas[year][area].at['Geometric Aggregtion', 'CShareD']
    dictCorrAreasDF[year].at[area,'AI vs RSI'] = dictCorrAreas[year][area].at['AI','RSI']
    dictCorrAreasDF[year].at[area,'AI vs F-Measure'] = dictCorrAreas[year][area].at['AI','FMeasure']
    dictCorrAreasDF[year].at[area,'CShareD vs Output'] = dictCorrAreas[year][area].at['CShareD','Output']
    dictCorrAreasDF[year].at[area,'DShareC vs CShareD'] = dictCorrAreas[year][area].at['DShareC','CShareD']

    #dictSpearmanCorrAreasDF[year].at[area,'AI vs Output'] = dictSpearmanCorrAreas[year][area].at['AI','Output']
    dictSpearmanCorrAreasDF[year].at[area,'Output'] =  df_areas.at[year,area]
    dictSpearmanCorrAreasDF[year].at[area,'AI vs DShareC'] = dictSpearmanCorrAreas[year][area].at['AI', 'DShareC']
    dictSpearmanCorrAreasDF[year].at[area,'AI vs CShareD'] = dictSpearmanCorrAreas[year][area].at['AI','CShareD']
    #dictSpearmanCorrAreasDF[year].at[area,'F-Measure vs Output'] = dictSpearmanCorrAreas[year][area].at['FMeasure','Output']
    dictSpearmanCorrAreasDF[year].at[area,'F-Measure vs DShareC'] = dictSpearmanCorrAreas[year][area].at['FMeasure', 'DShareC']
    dictSpearmanCorrAreasDF[year].at[area,'F-Measure vs CShareD'] = dictSpearmanCorrAreas[year][area].at['FMeasure', 'CShareD']
    #dictSpearmanCorrAreasDF[year].at[area,'Geometric Aggregtion vs Output'] = dictSpearmanCorrAreas[year][area].at['Geometric Aggregtion','Output']
    dictSpearmanCorrAreasDF[year].at[area,'Geometric Aggregtion vs DShareC'] = dictSpearmanCorrAreas[year][area].at['Geometric Aggregtion', 'DShareC']
    dictSpearmanCorrAreasDF[year].at[area,'Geometric Aggregtion vs CShareD'] = dictSpearmanCorrAreas[year][area].at['Geometric Aggregtion', 'CShareD']
    dictSpearmanCorrAreasDF[year].at[area,'AI vs RSI'] = dictSpearmanCorrAreas[year][area].at['AI','RSI']
    dictSpearmanCorrAreasDF[year].at[area,'AI vs F-Measure'] = dictSpearmanCorrAreas[year][area].at['AI','FMeasure']
    dictSpearmanCorrAreasDF[year].at[area,'CShareD vs Output'] = dictSpearmanCorrAreas[year][area].at['CShareD','Output']
    dictSpearmanCorrAreasDF[year].at[area,'DShareC vs CShareD'] = dictSpearmanCorrAreas[year][area].at['DShareC','CShareD']

    #dictCorrAreasDFAGMeans[year].at[area,'A-Measure vs Output'] = dictCorrAreas[year][area].at['AMeasure','Output']
    dictCorrAreasDFAGMeans[year].at[area,'AMean vs DShareC'] = dictCorrAreas[year][area].at['AMean', 'DShareC']
    dictCorrAreasDFAGMeans[year].at[area,'AMean vs CShareD'] = dictCorrAreas[year][area].at['AMean','CShareD']
    dictCorrAreasDFAGMeans[year].at[area,'AMean vs F-Measure'] = dictCorrAreas[year][area].at['AMean','FMeasure']
    #dictCorrAreasDFAGMeans[year].at[area,'G-Measure vs Output'] = dictCorrAreas[year][area].at['GMeasure','Output']
    dictCorrAreasDFAGMeans[year].at[area,'GMean vs DShareC'] = dictCorrAreas[year][area].at['GMean', 'DShareC']
    dictCorrAreasDFAGMeans[year].at[area,'GMean vs CShareD'] = dictCorrAreas[year][area].at['GMean', 'CShareD']
    dictCorrAreasDFAGMeans[year].at[area,'GMean vs F-Measure'] = dictCorrAreas[year][area].at['GMean','FMeasure']

    #dictSpearmanCorrAreasDFAGMeans[year].at[area,'A-Measure vs Output'] = dictSpearmanCorrAreas[year][area].at['AMeasure','Output']
    dictSpearmanCorrAreasDFAGMeans[year].at[area,'AMean vs DShareC'] = dictSpearmanCorrAreas[year][area].at['AMean', 'DShareC']
    dictSpearmanCorrAreasDFAGMeans[year].at[area,'AMean vs CShareD'] = dictSpearmanCorrAreas[year][area].at['AMean','CShareD']
    dictSpearmanCorrAreasDFAGMeans[year].at[area,'AMean vs F-Measure'] = dictSpearmanCorrAreas[year][area].at['AMean','FMeasure']
    #dictSpearmanCorrAreasDFAGMeans[year].at[area,'G-Measure vs Output'] = dictSpearmanCorrAreas[year][area].at['GMeasure','Output']
    dictSpearmanCorrAreasDFAGMeans[year].at[area,'GMean vs DShareC'] = dictSpearmanCorrAreas[year][area].at['GMean', 'DShareC']
    dictSpearmanCorrAreasDFAGMeans[year].at[area,'GMean vs CShareD'] = dictSpearmanCorrAreas[year][area].at['GMean', 'CShareD']
    dictSpearmanCorrAreasDFAGMeans[year].at[area,'GMean vs F-Measure'] = dictSpearmanCorrAreas[year][area].at['GMean','FMeasure']

    dictYearlyData[year][area] = dfTemp


In [ ]:
import os
import numpy as np
from scipy.stats import shapiro
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt


text= '''
  <!DOCTYPE html>
  <html>
  <body>

  <br>
  <br>
  <h1>ScatterPlots</h1>
  <br>
  <br>
  <ul>
  '''


dicTests = {}
average = 0
os.makedirs('scatterplots', exist_ok = True)

for year in years:
  if year != 2024:
    continue
  os.makedirs('scatterplots/'+str(year), exist_ok = True)
  textScatterplots = '''
  <!DOCTYPE html>
  <html>
  <body>

  <br>
  <br>
  <h1>ScatterPlots '''+str(year)+'''+</h1>
  <br>
  <br>

  '''

  dicTests[year] = pd.DataFrame(index = areas)
  for area in areas:
    df = dictYearlyData[year][area].sort_values('Output', ascending=False)
    ## Select a subset indicators
    df = df[['Output','DShareC', 'CShareD', 'AI', 'FMeasure', 'RSI']]
    # Add the table to the html
    textScatterplots = textScatterplots +'''<br> <br> <h1>'''+ str(year) + " - "+area +'''</h1> <br><br>'''

    cm = sns.light_palette("lightblue", as_cmap=True)
    s = df.style.background_gradient(cmap=cm)
    textScatterplots = textScatterplots + s.to_html()
    textScatterplots = textScatterplots + '''<br><br>'''
    ## Create a pairplot with plotly
    fig = px.scatter_matrix(df,
                            dimensions=df.columns,
                            symbol=df.index,
                            title=str(year) + " - "+area)
    fig.update_traces(diagonal_visible=False)
    areaS = area.replace('/', '')
    figName = 'scatterplots/'+str(year)+'/'+str(year)+'_'+areaS
    figNamehtml = figName+'.html'
    #html
    fig.write_html(figNamehtml,
                   full_html=False,
                   include_plotlyjs='cdn')

    textScatterplots = textScatterplots + '''<a href="'''+figNamehtml.replace('scatterplots/', '')+'''">Plotly - '''+area+'''</a>'''

    textScatterplots = textScatterplots + '''
    <br><br>
    '''
    # Set a global font scale
    sns.set_theme(font_scale=1.6)
    #Create a pairplot with seaborn
    scatter_plot  = sns.pairplot(df)
    scatter_fig = scatter_plot.fig
    scatter_fig.savefig(figName+'.png')
    textScatterplots = textScatterplots + '''
    <img src="'''+figName.replace('scatterplots/', '')+'''.png" alt="'''+str(year) + ''' - '''+area+'''">
    <br><br>'''
    #Intent closing the figure
    plt.close()

    dicTests[year].at[area,'Number of Countries'] = len(df)
    for indicator in df.columns:
      stat, p = shapiro(df[indicator])
      if p > 0.05:
        dicTests[year].at[area, indicator +'-Shapiro' ] = 1
      else:
        dicTests[year].at[area, indicator +'-Shapiro' ] = 0
  ave =  dicTests[year]['Number of Countries'].mean()
  print(year, ' - ' , ave)
  average += ave
  #close html and save it
  textScatterplots = textScatterplots + '''  </body></html>'''

  file = open("scatterplots/scatterPlots_"+str(year)+".html","w")
  file.write(textScatterplots)
  file.close()
  text = text + '''<li><a href="'''+"scatterplots/scatterPlots_"+str(year)+".html"+'''">'''+str(year)+'''</a></li>'''





text = text + '''</ul> </body></html>'''
file = open("scatterplots.html","w")
file.write(text)
file.close()

average = average / len(years)
print('Average', average)


In [ ]:
import plotly.express as px

fig = px.scatter_matrix(dfTemp,
    dimensions=dfTemp.columns,
                        symbol=dfTemp.index,
    title="Scatter matrix") # remove underscore
fig.update_traces(diagonal_visible=False)


fig.show()

In [ ]:
import plotly.express as px

fig = px.scatter_matrix(dfTemp,
    dimensions=dfTemp.columns,
                        symbol=dfTemp.index,
    title="Scatter matrix") # remove underscore
fig.update_traces(diagonal_visible=False)

fig.update_layout(
    xaxis=dict(type='log'),
    yaxis=dict(type='log'),)


In [ ]:
dfTemp.head()

In [ ]:
# x and y given as array_like objects
import plotly.express as px
fig = px.scatter(x=dfTemp['CShareD'], y=dfTemp['DShareC'])
fig.update_layout(
    xaxis=dict(type='log'),
    yaxis=dict(type='log'),)
fig.show()

# Create and html report

In [ ]:
text = '''<h1>Manuscript figures and tables - Global Context</h1>

Pearson correlations in the year 2024 with countries as entities
'''

In [ ]:
import seaborn as sns

def style_negative(v, props=''):
    return props if v < 0 else None

df = dictCorrAreasDF[2024].sort_values('F-Measure vs DShareC', ascending=False)
pd.set_option("styler.format.precision", 4)
cm = sns.light_palette("lightblue", as_cmap=True)

if(len(df) > 50):
  s = df.head(50).style.background_gradient(cmap=cm, vmin=0, vmax=1)
else:
  s = df.style.background_gradient(cmap=cm, vmin=0, vmax=1)
s.applymap(style_negative, props='color:red;')
text = text + s.to_html()
df.to_csv('pearson_correlation_AI_FMeasure.csv')

In [ ]:
import seaborn as sns

df = dictCorrAreasDFAGMeans[2024]
pd.set_option("styler.format.precision", 4)
cm = sns.light_palette("lightblue", as_cmap=True)

if(len(df) > 50):
  s = df.head(50).style.background_gradient(cmap=cm, vmin=0, vmax=1)
else:
  s = df.style.background_gradient(cmap=cm, vmin=0, vmax=1)
s.applymap(style_negative, props='color:red;')
text = text + s.to_html()

In [ ]:
text = text + '''
<h1>Spearman correlations in the year 2024 with countries as entities</h1>
'''

In [ ]:
import seaborn as sns

df = dictSpearmanCorrAreasDF[2024].sort_values('F-Measure vs DShareC', ascending=False)
pd.set_option("styler.format.precision", 4)
cm = sns.light_palette("lightblue", as_cmap=True)

#if(len(df) > 100):
#  s = df.head(100).style.background_gradient(cmap=cm)
#else:
#  s = df.style.background_gradient(cmap=cm)

s = df.style.background_gradient(cmap=cm, vmin=0, vmax=1)
s.applymap(style_negative, props='color:red;')
text = text + s.to_html()
df.to_csv('Spearman_correlation_AI_FMeasure.csv')

In [ ]:
import seaborn as sns

df = dictSpearmanCorrAreasDFAGMeans[2024]
pd.set_option("styler.format.precision", 4)
cm = sns.light_palette("lightblue", as_cmap=True)

#if(len(df) > 100):
#  s = df.head(100).style.background_gradient(cmap=cm)
#else:
#  s = df.style.background_gradient(cmap=cm)

s = df.style.background_gradient(cmap=cm, vmin=0, vmax=1)
s.applymap(style_negative, props='color:red;')
text = text + s.to_html()

### Charts

In [ ]:
dictYearlyData[year][area]

In [ ]:
import plotly.express as px
import plotly.graph_objs as go

# Tnaks to user171780 in https://stackoverflow.com/questions/61494278/plotly-how-to-make-a-figure-with-multiple-lines-and-shaded-area-for-standard-de
def line(error_y_mode=None, **kwargs):
    """Extension of `plotly.express.line` to use error bands."""
    ERROR_MODES = {'bar','band','bars','bands',None}
    if error_y_mode not in ERROR_MODES:
        raise ValueError(f"'error_y_mode' must be one of {ERROR_MODES}, received {repr(error_y_mode)}.")
    if error_y_mode in {'bar','bars',None}:
        fig = px.line(**kwargs)
    elif error_y_mode in {'band','bands'}:
        if 'error_y' not in kwargs:
            raise ValueError(f"If you provide argument 'error_y_mode' you must also provide 'error_y'.")
        figure_with_error_bars = px.line(**kwargs)
        fig = px.line(**{arg: val for arg,val in kwargs.items() if arg != 'error_y'})
        for data in figure_with_error_bars.data:
            x = list(data['x'])
            y_upper = list(data['y'] + data['error_y']['array'])
            y_lower = list(data['y'] - data['error_y']['array'] if data['error_y']['arrayminus'] is None else data['y'] - data['error_y']['arrayminus'])
            color = f"rgba({tuple(int(data['line']['color'].lstrip('#')[i:i+2], 16) for i in (0, 2, 4))},.3)".replace('((','(').replace('),',',').replace(' ','')
            fig.add_trace(
                go.Scatter(
                    x = x+x[::-1],
                    y = y_upper+y_lower[::-1],
                    fill = 'toself',
                    fillcolor = color,
                    line = dict(
                        color = 'rgba(255,255,255,0)'
                    ),
                    hoverinfo = "skip",
                    showlegend = False,
                    legendgroup = data['legendgroup'],
                    xaxis = data['xaxis'],
                    yaxis = data['yaxis'],
                )
            )
        # Reorder data as said here: https://stackoverflow.com/a/66854398/8849755
        reordered_data = []
        fig.update_layout(hovermode="x")
        for i in range(int(len(fig.data)/2)):
            reordered_data.append(fig.data[i+int(len(fig.data)/2)])
            reordered_data.append(fig.data[i])
        fig.data = tuple(reordered_data)
    return fig

# This function receives a dataframe containing de correlations
def makeMeanStdSeries(df, cols):
  dfResult = pd.DataFrame(columns=['Name', 'Year', 'Mean', 'Std'])
  for year in years:
    for col in cols:
      dft = df[year][[col]]
      dftDesception = dft.describe()
      dfResult.loc[len(dfResult.index)] = [col, year, dftDesception.at['mean', col], dftDesception.at['std', col]]
  return dfResult

def makeSeries(df, col):
  dfResult = pd.DataFrame(columns=['Domain Area', 'Year', col])
  for year in years:
    dft = df[year][[col]]
    for area in areas:
      dfResult.loc[len(dfResult.index)] = [area, year, dft.at[area, col]]#Agregar un renglon
  return dfResult

def makeAreaSeries(df, col):

  dfT = df[2024][[col]]
  minV = dfT.min().values[0]
  areaMin = dfT[dfT[col] == minV].index[0]
  maxV = dfT.max().values[0]
  areaMax = dfT[dfT[col] == maxV].index[0]

  dfResult = pd.DataFrame(columns=['Domain', 'Year', col])
  for year in years:
    dfAreaMax = df[year][[col]].loc[[areaMax]]
    dfAreaMax['Year'] = year
    dfAreaMax['Domain'] = areaMax
    dfResult = pd.concat([dfResult, dfAreaMax])
    dfAreaMin = df[year][[col]].loc[[areaMin]]
    dfAreaMin['Year'] = year
    dfAreaMin['Domain'] = areaMin
    dfResult = pd.concat([dfResult, dfAreaMin])
  return dfResult


In [ ]:
import plotly.graph_objects as go

# Get the dataframe for the current year and sort it by 'Output'
df_plot = dictSpearmanCorrAreasDF[year].sort_values('F-Measure vs CShareD', ascending=False)

# Create a new figure
fig = go.Figure()

# Add traces for 'F-Measure vs DShareC'
fig.add_trace(go.Bar(
    x=df_plot.index, # Areas on the x-axis
    y=df_plot['F-Measure vs DShareC'],
    name='F-Measure vs DShareC'
))

# Add traces for 'F-Measure vs CShareD'
fig.add_trace(go.Bar(
    x=df_plot.index, # Areas on the x-axis
    y=df_plot['F-Measure vs CShareD'],
    name='F-Measure vs CShareD'
))

# Update layout for better readability
fig.update_layout(
    title=f'F-Measure Correlations per Area (Year {year}) - Ordered by Output',
    xaxis_title='Domain Area',
    yaxis_title='Spearman Correlation Value',
    barmode='group', # Group bars for each area
    height=600
)

fig.show()


figName = 'figuras/Spearman_correlation_DomainAreas_FMeasureDShareC_CShareD'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''


In [ ]:
df = makeSeries(dictCorrAreasDF, 'F-Measure vs DShareC')
df = df[df['Year'].isin([1980, 1990, 2000, 2010, 2020])]
fig = px.box(df, y="F-Measure vs DShareC", x="Year", color="Year",  points="all", title = 'Pearson Correlation',
          hover_data=df.columns)
fig.update_layout(
    font=dict(
        size=18,
    )
)
fig.show()

figName = 'figuras/pearson_correlation_longitudinal_fmeasure_nshare'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=True,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
df = makeSeries(dictSpearmanCorrAreasDF, 'F-Measure vs DShareC')
df = df[df['Year'].isin([1980, 1990, 2000, 2010, 2020])]
fig = px.box(df, y="F-Measure vs DShareC", x="Year", color="Year",  points="all", title = 'Spearman Correlation',
          hover_data=df.columns)
fig.update_layout(
    font=dict(
        size=18,
    ))
fig.show()

figName = 'figuras/spearman_correlation_longitudinal_fmeasure_nshare'
figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
df = makeSeries(dictCorrAreasDF, 'F-Measure vs CShareD')
df = df[df['Year'].isin([1980, 1990, 2000, 2010, 2020])]
fig = px.box(df, y="F-Measure vs CShareD", x="Year", color="Year",  points="all", title = 'Pearson Correlation',
          hover_data=df.columns)
fig.update_layout(
    font=dict(
        size=18,
    ))
fig.show()

figName = 'figuras/pearson_correlation_longitudinal_fmeasure_wshare'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
df =makeSeries(dictSpearmanCorrAreasDF, 'F-Measure vs DShareC')
df=df[df['Year']==2024]
df = df[['Domain Area', 'F-Measure vs DShareC']]
df = df.sort_values(by='F-Measure vs DShareC', ascending=False)
df

In [ ]:
df.describe()

In [ ]:
df = makeSeries(dictSpearmanCorrAreasDF, 'F-Measure vs CShareD')
df = df[df['Year'].isin([1980, 1990, 2000, 2010, 2020])]
fig = px.box(df, y="F-Measure vs CShareD", x="Year", color="Year",  points="all", title = 'Spearman Correlation',
          hover_data=df.columns)
fig.update_layout(
    font=dict(
        size=18,
    ))
fig.show()

figName = 'figuras/spearman_correlation_longitudinal_fmeasure_wshare'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')

text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
dfT = makeMeanStdSeries(dictCorrAreasDF, ['F-Measure vs DShareC', 'F-Measure vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'bands', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Pearson Correlation',
        markers = '.',
    )
fig.show()

figName = 'figuras/pearson_correlation_evolution_fmeasure'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
dfT = makeMeanStdSeries(dictSpearmanCorrAreasDF, ['F-Measure vs DShareC', 'F-Measure vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'bands', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Spearman Correlation',
        markers = '.',
    )
fig.show()

figName = 'figuras/spearman_correlation_evolution_fmeasure'


figNamehtml = figName+'.html'
#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
dfT = makeMeanStdSeries(dictCorrAreasDF, ['Geometric Aggregtion vs DShareC', 'Geometric Aggregtion vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'bands', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Pearson Correlation',
        markers = '.',
    )
fig.show()

figName = 'figuras/pearson_correlation_evolution_Geometric_aggregtion'


figNamehtml = figName+'.html'
#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
dfT = makeMeanStdSeries(dictCorrAreasDF, ['AI vs DShareC', 'AI vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'band', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Pearson Correlation',
        markers = '.',
    )
fig.show()

figName = 'figuras/pearson_correlation_evolution_AI'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
dfT = makeMeanStdSeries(dictSpearmanCorrAreasDF, ['AI vs DShareC', 'AI vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'band', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Spearman Correlation',
        markers = '.',
    )
fig.show()

figName = 'figuras/spearman_correlation_evolution_AI'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
df = makeSeries(dictCorrAreasDF, 'AI vs CShareD')
df = df[df['Year'].isin([1980, 1990, 2000, 2010, 2020])]
fig = px.box(df, y="AI vs CShareD", x="Year", color="Year",  points="all", title = 'Pearson Correlation',
          hover_data=df.columns)
fig.show()

figName = 'figuras/pearson_correlation_longitudinal_AI_wshare'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:

dfT = makeMeanStdSeries(dictCorrAreasDFAGMeans, ['AMean vs DShareC', 'AMean vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'bands', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Pearson Correlation',
        markers = '.',
    )
fig.show()

figName = 'figuras/pearson_correlation_evolution_A-Measure'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:

dfT = makeMeanStdSeries(dictSpearmanCorrAreasDFAGMeans, ['AMean vs DShareC', 'AMean vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'bands', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Spearman Correlation',
        markers = '.',
    )
fig.show()


figName = 'figuras/spearman_correlation_evolution_A-Measure'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:

dfT = makeMeanStdSeries(dictCorrAreasDFAGMeans, ['GMean vs DShareC', 'GMean vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'bands', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Pearson Correlation',
        markers = '.',
    )
fig.show()


figName = 'figuras/pearson_correlation_evolution_G-Measure'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
dfT = makeMeanStdSeries(dictSpearmanCorrAreasDFAGMeans, ['GMean vs DShareC', 'GMean vs CShareD'])

fig = line(
        data_frame = dfT,
        x = 'Year',
        y = 'Mean',
        error_y = 'Std',
        error_y_mode = 'bands', # Here you say `band` or `bar`.
        color = 'Name',
        title = 'Spearman Correlation',
        markers = '.',
    )
fig.show()


figName = 'figuras/spearman_correlation_evolution_G-Measure'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
dictCorrAreasDF[year]

In [ ]:
import plotly.express as px

col = 'F-Measure vs DShareC'
df = makeAreaSeries(dictCorrAreasDF, col)

fig = px.line(df, x="Year", y=col, color='Domain', title = 'Pearson Correlation', markers = '.')
fig.show()


figName = 'figuras/pearson_correlation_evolution_FMeasureDShareC2Areas'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''


In [ ]:
import plotly.express as px

col = 'F-Measure vs DShareC'
df = makeAreaSeries(dictSpearmanCorrAreasDF, col)

fig = px.line(df, x="Year", y=col, color='Domain', title = 'Spearman Correlation', markers = '.')
fig.show()

figName = 'figuras/spearman_correlation_best_worst_fmeasure_nshare'

figNamehtml = figName+'.html'
#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
col = 'AI vs CShareD'
df = makeAreaSeries(dictSpearmanCorrAreasDF, col)

fig = px.line(df, x="Year", y=col, color='Domain', title = 'Spearman Correlation', markers = '.')
fig.show()

figName = 'figuras/spearman_correlation_best_worst_ai_wshare'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictCorrAreasDF, 'F-Measure vs DShareC')
fig = px.line(df, x="Year", y="F-Measure vs DShareC", color='Domain Area',  title = 'Pearson Correlation', markers = '.')
fig.show()

figName = 'figuras/pearson_correlation_fmeasure_DShareC_alldomains'


figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictCorrAreasDF, 'F-Measure vs CShareD')
fig = px.line(df, x="Year", y="F-Measure vs CShareD", color='Domain Area', title = 'Pearson Correlation', markers = '.')
fig.show()

figName = 'figuras/pearson_correlation_fmeasure_CShareD_alldomains'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictCorrAreasDF, 'AI vs DShareC')
fig = px.line(df, x="Year", y="AI vs DShareC", color='Domain Area',  title = 'Pearson Correlation', markers = '.')
fig.show()


figName = 'figuras/pearson_correlation_AI_DShareC_alldomains'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictCorrAreasDF, 'AI vs CShareD')
fig = px.line(df, x="Year", y="AI vs CShareD", color='Domain Area',  title = 'Pearson Correlation', markers = '.')
fig.show()

figName = 'figuras/pearson_correlation_AI_CShareD_alldomains'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictSpearmanCorrAreasDF, 'F-Measure vs DShareC')
fig = px.line(df, x="Year", y="F-Measure vs DShareC", color='Domain Area', title = 'Spearman Correlation', markers = '.')
fig.show()

figName = 'figuras/Spearman_correlation_fmeasure_DShareC_alldomains'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictSpearmanCorrAreasDF, 'F-Measure vs CShareD')
fig = px.line(df, x="Year", y="F-Measure vs CShareD", color='Domain Area', title = 'Spearman Correlation', markers = '.')
fig.show()

figName = 'figuras/Spearman_correlation_fmeasure_CShareD_alldomains'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictSpearmanCorrAreasDF, 'AI vs DShareC')
fig = px.line(df, x="Year", y="AI vs DShareC", color='Domain Area', title = 'Spearman Correlation', markers = '.')
fig.show()

figName = 'figuras/Spearman_correlation_AI_DShareC_alldomains'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

In [ ]:
import plotly.express as px

df = makeSeries(dictSpearmanCorrAreasDF, 'AI vs CShareD')
fig = px.line(df, x="Year", y="AI vs CShareD", color='Domain Area', title = 'Spearman Correlation', markers = '.')
fig.show()

figName = 'figuras/Spearman_correlation_AI_CShareD_alldomains'

figNamehtml = figName+'.html'

#html
fig.write_html(figNamehtml,
              full_html=False,
              include_plotlyjs='cdn')
text = text + '''<iframe src="'''+figNamehtml+'''" width="100%" height="100%"></iframe>'''

# Tables

In [ ]:
print(year)
dictCorrAreas[year][area].style.format(precision=4)

In [ ]:
dictCorrAreasDF[2024].style.format(precision=4)

In [ ]:
dictCorrAreasDF[2022].style.format(precision=4)

In [ ]:
dictCorrAreasDF[year].describe().style.format(precision=4)

In [6]:
dictCorrAreasDF[1980].style.format(precision=4)

NameError: name 'dictCorrAreasDF' is not defined

In [ ]:
dictSpearmanCorrAreasDF[year].style.format(precision=3)
dictSpearmanCorrAreasDF[year].to_csv('spearman_correlation_2024.csv')

In [ ]:
dictSpearmanCorrAreasDF[year].describe().style.format(precision=4)

In [ ]:
dictCorrAreasDFAGMeans[year].style.format(precision=4)

In [ ]:
dictCorrAreasDFAGMeans[year].describe().style.format(precision=4)

In [7]:
dictSpearmanCorrAreasDFAGMeans[year].style.format(precision=4)

NameError: name 'dictSpearmanCorrAreasDFAGMeans' is not defined

In [8]:
dictSpearmanCorrAreasDFAGMeans[year].describe().style.format(precision=4)

NameError: name 'dictSpearmanCorrAreasDFAGMeans' is not defined

In [9]:
text = text + '''</div>'''
file = open("manuscriptFiguresAndTables.html","w")
file.write(text)
file.close()

NameError: name 'text' is not defined